# Shared setup for causal tracing
Run on a Colab GPU. This loads the exact GPT-2 XL revision and precision used for the baseline. It does not rerun 200 edits.

The Python 3.11 model lives inside the subprocess below, not in Colab's Python 3.13 kernel. Add tracing code to the same `EXPERIMENT` block to reuse `model`, `tokenizer`, and `mt` in that process. A new execution reloads the model. This starter has not yet been run on a GPU.


In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
os.environ['MPLBACKEND'] = 'Agg'
ROOT = Path('/content/nlp544')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/Gogireddy007/Multifact-Rome.git',str(ROOT)],check=True)

def stream(command):
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
        for line in p.stdout:
            print(line,end='',flush=True)
        if p.wait():
            raise RuntimeError('Command failed; see output above.')

stream([sys.executable,'-u',str(ROOT/'scripts/setup.py')])
PYTHON = str(ROOT/'.venv/bin/python')


In [ ]:
EXPERIMENT = r"""
import sys
sys.path.insert(0, '/content/nlp544/scripts')
from model_setup import load_model_and_tokenizer
model, tokenizer = load_model_and_tokenizer()
from experiments.causal_trace import ModelAndTokenizer
mt = ModelAndTokenizer(model=model, tokenizer=tokenizer)
print(mt)
print('MODEL READY')
# Soham: add causal tracing experiments here using mt.
# Official helpers: calculate_hidden_flow, plot_trace_heatmap,
# collect_embedding_std. Choose known facts and calibrate noise.
"""
stream([PYTHON, '-u', '-c', EXPERIMENT])
